# Binary DDoS Detection

Goal: train and evaluate machine learning models to distinguish between BENIGN and DDoS network traffic using the CIC-IDS-2017 dataset.

In [ ]:
## 1. Load dataset
import pandas as pd
df=pd.read_csv('../data/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv')
df.head()

In [ ]:
## 2. Prepare features and labels
df.columns = df.columns.str.strip()
print(df["Label"].unique())
X = df.drop(columns=["Label"])

y = df["Label"]

y_binary = y.apply(lambda x: 0 if x == "BENIGN" else 1)

In [ ]:
## 3. Explore labels
print(X.shape)
print(y_binary.shape)

In [ ]:
#percentagem de ataques/normais
y_binary.value_counts(normalize=True)

In [ ]:
## 4. Train/test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_binary,
    test_size=0.2,
    random_state=42,
    stratify=y_binary
)

In [ ]:
print(X_train.shape)
print(X_test.shape)
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))


In [ ]:
for x in list(df.columns): 
    print(x)

In [ ]:
## 5. Decision Tree model
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(random_state=1)
model.fit(X_train, y_train)

In [ ]:
import numpy as np

# substitui inf e -inf por NaN, depois remove as linhas com NaN
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)

print("NaN em X_train:", X_train.isna().sum().sum())

X_train = X_train.dropna()
y_train = y_train[X_train.index]

X_test = X_test.dropna()
y_test = y_test[X_test.index]

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

pred= model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, pred))

print("Precision:", precision_score(y_test, pred))

print("Recall:", recall_score(y_test, pred))

print("F1-score:", f1_score(y_test, pred))

In [ ]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, pred))

In [ ]:
## 6. Feature importance
import pandas as pd

importances = pd.DataFrame({
    "feature": X.columns,
    "importance": model.feature_importances_
})

importances.sort_values("importance", ascending=False).head(15)

## 7. Conclusions

Both Random Forest and Decision Tree achieved very high performance, with almost perfect accuracy, precision, recall and F1-score. The confusion matrix showed that the models made only a very small number of incorrect predictions.

However, this experiment is still limited because it uses only one file and only one attack type. Therefore, the task is relatively simple compared to a real intrusion detection system.

The next step is to use multiple CIC-IDS-2017 CSV files and move from binary classification to multiclass classification, where the model will try to distinguish between BENIGN traffic and different types of attacks.